## Imports and data loading

In [1]:
import math
import random
import time

import pandas as pd
import numpy as np

from scipy import sparse
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics.pairwise import cosine_similarity

from sparselsh import LSH

## Content-based, regression

In [2]:
# load food.com data
directory = 'data/food.com'
df_recipe_rating = pd.read_csv(f'{directory}/recipe_ratings.csv')
df_recipe = pd.read_csv(f'{directory}/recipe.csv')

In [3]:
df_recipe.head()

,recipe_id,name,minutes,n_steps,n_ingredients,calories,fat,sugar,sodium,protein,saturated,carbs
0,137739,arriba baked winter squash mexican style,55,11,7,51.5,0.0,13.0,0.0,2.0,0.0,4.0
1,31490,a bit different breakfast pizza,30,9,6,173.4,18.0,0.0,17.0,22.0,35.0,1.0
2,112140,all in the kitchen chili,130,6,13,269.8,22.0,32.0,48.0,39.0,27.0,5.0
3,59389,alouette potatoes,45,11,11,368.1,17.0,10.0,2.0,14.0,8.0,20.0
4,44061,amish tomato ketchup for canning,190,5,8,352.9,1.0,337.0,23.0,3.0,0.0,28.0


In [4]:
df_recipe_rating

,user_id,recipe_id,rating
0,2046,4684,5.0
1,2046,517,5.0
2,1773,7435,5.0
3,1773,278,4.0
4,2046,3431,5.0
...,...,...,...
698896,926904,457971,5.0
698897,2002312797,27208,5.0
698898,1290903,131607,5.0
698899,226867,363072,5.0


In [ ]:
#EXERCISE: Build a content-based recommender system that uses linear regression 
#          to predict ratings.
#          Try it out on users with a high number of ratings.   
#          Try some train-test split to evaluate performance.

# find users with many ratings AND high variance
user_stats = df_recipe_rating.groupby('user_id')['rating'].agg(['count', 'std'])                                 
top_users = user_stats[(user_stats['count'] > 100) & (user_stats['std'] > 1.0)]                                  
top_users = top_users.sort_values('std', ascending=False)                                                        
print(top_users.head(10))                                                                                        

# pick a user with a high number of ratings
target_id = top_users.index[0]                                                                                 
print(f"Selected user: {target_id}")                                                                             

# select the ratings of a specific user 
df_user = df_recipe_rating[df_recipe_rating['user_id']==target_id][['recipe_id','rating']] # recipes for which the user `target_id` has given ratings
df_rec = pd.merge(df_user, df_recipe, on='recipe_id', how='inner') # merge with the recipe data to get the features of the recipes  

df_user['rating'].value_counts()
df_user['rating'].describe()


         count       std
user_id                 
202661     111  2.283079
220195     127  2.069261
904483     120  2.025036
841895     152  2.021570
502302     192  1.953624
184530     205  1.949236
583193     135  1.935471
109110     106  1.915112
207176     462  1.900351
201584     119  1.860924
Selected user: 202661


count    111.000000
mean       3.225225
std        2.283079
min        0.000000
25%        0.000000
50%        5.000000
75%        5.000000
max        5.000000
Name: rating, dtype: float64

In [ ]:
# define features
features = ['minutes', 'n_steps', 'n_ingredients', 'calories', 'fat', 'sugar', 'sodium', 'protein', 'saturated', 'carbs']

# split training and test
# 80/20 split
train_size = int(0.8 * len(df_rec))
train_df = df_rec[:train_size]
test_df = df_rec[train_size:]

# fit on the training, test on the rest
model = LinearRegression()
model.fit(train_df[features], train_df['rating'])

predictions = model.predict(test_df[features])

# evaluate performance
from sklearn.metrics import mean_squared_error
mse = mean_squared_error(test_df['rating'], predictions)

print(f'Mean Squared Error: {mse}')

baseline_prediction = train_df['rating'].mean()
baseline_mse = mean_squared_error(test_df['rating'], [baseline_prediction] * len(test_df))
print(f'Baseline Mean Squared Error: {baseline_mse}')

Mean Squared Error: 5.940309054735236
Baseline Mean Squared Error: 5.90679460114984


### Reflections

- seems like recommending food based on nutritional values does not generally work (maybe except for bodybuilders)

## Content-based, KNN (with LSH)

In [7]:
directory = 'data/movielens/ml-latest-small'
#directory = 'data/movielens/ml-latest' #change into this one for the full dataset (slow)

df_movies = pd.read_csv(f'{directory}/movies.csv')
df_ratings = pd.read_csv(f'{directory}/ratings.csv')
df_tags = pd.read_csv(f'{directory}/tags.csv')

#transform tags such that they are lower-case, single-word tokens
df_tags['tag'] = df_tags['tag'].apply(lambda x: str(x).lower().replace(' ', '_'))

In [ ]:
df_movies.head()

In [ ]:
df_tags.head()

### Step1: Calculate item profiles

In [ ]:
# calculates the lexicon of most frequent tags.
tag_frequency_threshold = 5 # increase number to filter
df_lexicon = ... # get a dataframe with tags and respective counts

# discard movies with no tags
...

# you can drop the userId and timestamp columns because we don't care who assigned the tag and when
...

In [ ]:
#calculate the sparse feature vector based on the TF-IDF of words in documents
#the TF-IDF vectors are saved as sparse representations into the dataframe
df_features = df_tags.groupby('movieId').agg(lambda x: ' '.join(x)).reset_index()
vectorizer = TfidfVectorizer(tokenizer=lambda x: x.split(' ')).fit(sorted(df_features['tag']))
vectorizer.vocabulary_
df_features['feature_vector'] = df_features['tag'].apply(lambda x : vectorizer.transform([x]))
df_features

### Step2: Index item profiles into LSH

In [ ]:
#index all item vectors into LSH
lsh = LSH(...)

#run an example query to the LSH
lsh.query(...)

### Step 3: Calculate user profile

In [ ]:
# restricts the ratings to the set of most popular movies (optional, not needed for content-based)
numratings_threshold = 0 #increase this number if you want to filter
df_item_popularity = df_ratings[['movieId','rating']].groupby('movieId').count().reset_index()
df_item_popularity.columns = ['movieId','count'] 
df_item_popularity = df_item_popularity.sort_values(by='count', ascending=False)
df_item_popularity = df_item_popularity[df_item_popularity['count'] >= numratings_threshold]
print(f'Number of movies reduced from {len(df_ratings.movieId.unique())} to {len(df_item_popularity.movieId.unique())}')
df_ratings = pd.merge(df_ratings, df_item_popularity, on='movieId', how='inner')[['userId', 'movieId', 'rating']]
df_ratings = df_ratings.sort_values(by='userId')

#rescale the ratings by the user's individual average 
df_ratings['rating_scaled'] = ...

df_ratings.head()

In [ ]:
# join ratings with movie feature vectors
df_profile = pd.merge(df_ratings, df_features[['movieId','feature_vector']],
              on='movieId')
#scaling feature vector by rating (this will take a few minutes)
df_profile['feature_vector_scaled'] = df_profile['rating_scaled'] * df_profile['feature_vector']
df_profile

In [ ]:
start = time.time()
#stack all sparse vectors of user's movies
df_user_vectors = df_profile[['userId', 'feature_vector_scaled']].groupby('userId').agg(sparse.vstack).reset_index()
#compute the average of the vectors without considering the zero entries (this will take a while)
df_user_vectors['feature_vector_scaled'] = df_user_vectors['feature_vector_scaled'].apply(lambda x: csr_matrix(np.nan_to_num(x.sum(axis=0)/x.getnnz(axis=0), 0)))
end = time.time()
print(end - start)
df_user_vectors

### Step 4: Rank potential recommendation candidates

In [ ]:
#pick a target user to provide recommendations to
idx = 42
target_userId = df_user_vectors.iloc[idx].userId

In [ ]:
#get user rating history
df_user_history = ...

#select candidate recommendations to user
df_recommendation = ...

In [ ]:
df_recommendation

In [ ]:
df_user_history.head(10)

In [ ]:
df_user_history.tail(10)

### Step 5: Predict ratings of candidate items

In [ ]:
#index all user vectors into LSH
df_usr = df_profile[df_profile['userId'] == target_userId]
lsh_usr = LSH(...)
lsh_usr.index(..., extra_data=[...]) # repeat for all users. Insert movieid and rating as extra data for future retrieval
lsh_usr

In [ ]:
# compute recommendation
df_recommendation = ...

## Collaborative filtering

In [ ]:
from surprise import SVD
from surprise import Reader
from surprise import Dataset
from surprise.model_selection import cross_validate
from surprise.prediction_algorithms.knns import KNNBasic

In [ ]:
directory = 'data/movielens/ml-latest-small'
#directory = 'data/movielens/ml-latest' #change into this one for the full dataset (slow)
df_ratings = pd.read_csv(f'{directory}/ratings.csv')
df_ratings.head()

In [ ]:
# initialize a data reader
reader = Reader(rating_scale=(1, 5))
# provide a dataset with userid, itemtid, and rating in order
data = Dataset.load_from_df(df_ratings[['userId','movieId','rating']], reader)

# surprise has also some built-in datasets that can be imported directly
#data = Dataset.load_builtin('ml-100k')

In [ ]:
# initialize a user-based K nearest neighbors implementation
...
# execute 5-fold cross-validation and measure RMSE and MAE
...